# 第24课 · 一条乘法链撑起整个深度学习——链式法则（chain rule），反向传播的种子

**目标**：复合函数 `f(g(x))` 的导数（derivative） = `f'(g(x)) · g'(x)`——一层层连乘。

**为什么对 Aurora 重要**：Month 2 的 `autograd（automatic differentiation，autograd）` 引擎在 `backward()` 里沿计算图（computation graph）连乘偏导（partial derivative，∂f/∂x）——`composite_derivative` 的分拆方式直接对应计算图上每条边的局部梯度（local gradient）。

← **上一课**　[L23 · 梯度](L23_gradients.ipynb)

> 上节课学习了 **梯度**：多元函数的"最陡上坡"方向，偏导与梯度向量的计算。  
> 本课将探讨 **链式法则**。

## 本课剧情：反向传播（backpropagation）靠什么？

神经网络有几十层。当你调整输入一点点，最终的损失会变多少？

答案是链式法则（chain rule）——函数套函数的求导规则：

```
y = f(g(x))  →  dy/dx = f'(g(x)) · g'(x)
```

直觉：内层把 x 的扰动放大（或缩小）g'(x) 倍，外层再把中间值的扰动放大 f'(g) 倍。两个放大倍数相乘，就是总的放大倍数。

**例子**：y = sin(x²)
- 外层 f(u) = sin(u)，f'(u) = cos(u)
- 内层 g(x) = x²，g'(x) = 2x
- dy/dx = cos(x²) · 2x

在 x=1 处：dy/dx = cos(1) · 2 ≈ 1.080

反向传播（backpropagation, BP）就是链式法则在计算图上的批量执行：  
从最后一层的误差出发，逐层向前"传递导数乘积"，直到每个参数的梯度都被算出来。

本课实现 `composite_derivative(x)` 并用数值差分验证，为 L56 反向传播铺路。

## 1. 链式法则：分层剥开，逐层相乘

**为什么叫"链"？**

把复合函数写成一条链：

```
x  →  g(x)=x²  →  f(u)=sin(u)  →  y=sin(x²)
```

导数沿链传播，每一步乘以那一层的"局部斜率"：

$$\frac{dy}{dx} = \frac{df}{du} \cdot \frac{dg}{dx} = f'(g(x)) \cdot g'(x)$$

**手算步骤**（y = sin(x²)）：

| 步骤 | 含义 | 结果 |
|---|---|---|
| 拆层 | 令 u = g(x) = x²，y = f(u) = sin(u) | — |
| 内层导数 | du/dx = g'(x) = 2x | 2x |
| 外层导数 | dy/du = f'(u) = cos(u) = cos(x²) | cos(x²) |
| 链式乘积 | dy/dx = cos(x²) · 2x | cos(x²)·2x |

**数值验证（x=1.3）**：解析值 = cos(1.69)·2.6 ≈ -0.309；中心差分结果应在误差 1e-9 内。

> **n 层推广**：每加一层，乘法链就多一项。反向传播（backprop）正是把这条乘法链写成矩阵-向量乘积，批量传递梯度。

## 实验入口：用数值变化观察函数

下面在 `x = 1.3` 处计算 `y = sin(x²)` 的数值导数，并与解析值 `cos(x²)·2x` 对比。关注 `numerical` 和 `analytical` 两列的误差大小。

In [ ]:
import numpy as np
x = 1.3
analytic = np.cos(x**2) * 2*x          # 链式法则
numeric = (np.sin((x+1e-5)**2) - np.sin((x-1e-5)**2)) / (2e-5)
print('链式法则:', round(analytic,5), ' 数值验证:', round(numeric,5))

> **[热身格 — 非链式法则]** 下面两格用最简单的多项式（`f(x)=x²`、`f(x)=x²+2x`）演练「数值差分」技术本身，
> 不涉及函数嵌套，因此**不是链式法则的应用**。目的是让你在接触复合函数前先确认差分公式的精度。
> 真正的链式法则示例从下一节（2. 实现 `composite_derivative`）开始。

## 动手观察：变化率不是一句口号

在一组 `x` 值上同时打印函数值和近似斜率，观察数值差分在每个点的精度。

In [ ]:
import numpy as np

def f(x):
    return x**2

xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 1e-3
slopes = (f(xs + h) - f(xs - h)) / (2 * h)

print('x =', xs)
print('f(x) =', f(xs))
print('近似斜率 =', np.round(slopes, 3))
print('理论斜率 2x =', 2 * xs)


## 代码实验：遍历不同位置，看斜率如何变化

下面遍历一列 `x`，把函数值和解析导数并排输出，确认链式法则在每个点都与数值差分吻合。

In [ ]:
import numpy as np

def f(x):
    return x**2 + 2*x

h = 1e-4
for x in np.linspace(-3, 3, 7):
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'x={x:5.2f} | f(x)={f(x):6.2f} | slope≈{slope:6.2f}')


## 2. ✏️ 实现 `composite_derivative(x)` —— y=sin(x²) 的导数

**推理路线**：
1. 拆层：外层 f(u)=sin(u)，内层 g(x)=x²，链式法则给出 dy/dx = f'(g(x)) · g'(x)
2. 分别求导：f'(u)=cos(u)，代入 u=x² 得 cos(x²)；g'(x)=2x
3. 相乘：结果是 cos(x²)·2x，NumPy 广播自动保证对数组输入也正确

**参考输入输出**：x=√(π/2) 时，x²=π/2，dy/dx=cos(π/2)·2x=0·√(2π)=0（sin(x²) 在该点取极值，导数为零）

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先把变量表补完整

写 `composite_derivative` 前明确三件事：
- 输入：`x`（标量或数组，单位弧度）
- 关键步骤：先算内层 `g(x) = x²`，再把外层导数 `cos(x²)` 和内层导数 `2x` 相乘
- 返回：与 `x` 等形状的导数数组，可直接和数值差分对比

In [ ]:
def composite_derivative(x):
    # ✏️ TODO: sin(x^2) 对 x 的导数
    # 提示：外层 f(u)=sin(u) → f'(u)=cos(u)；内层 g(x)=x² → g'(x)=2x
    # 链式法则：dy/dx = cos(x²) · 2x
    raise NotImplementedError("TODO: 实现 composite_derivative — 返回 cos(x**2) * 2*x")


In [ ]:
x = 1.3
numeric = (np.sin((x+1e-5)**2) - np.sin((x-1e-5)**2)) / (2e-5)
try:
    result = composite_derivative(x)
    assert abs(result - numeric) < 1e-6, (
        f"误差 {abs(result - numeric):.2e} 过大（应 < 1e-6）"
    )
    print("✅ 通过：你手算了一次链式法则 = 一次最小的反向传播。")
except (NotImplementedError, TypeError):
    print("⚠️  composite_derivative 尚未实现，请先完成 TODO 再运行此格。")


In [ ]:
# Aurora 连接：Month 2 的 autograd 引擎（L54 的 Value.backward）就是把本课的链式乘法自动化。
# 手动模拟一次最小的 "backward"：y = sin(x²) 的计算图只有两条边，各自缓存局部导数。
import numpy as np

x = 1.3
u = x**2                  # 前向：内层节点 g(x)
y = np.sin(u)             # 前向：外层节点 f(u)
local_du_dx = 2 * x       # 边 x→u 的局部导数（前向时就能缓存）
local_dy_du = np.cos(u)   # 边 u→y 的局部导数（前向时就能缓存）
grad = local_dy_du * local_du_dx   # backward：沿计算图的边连乘
print(f'dy/dx = {grad:.6f}（与链式法则手算 cos(x²)·2x 一致）')
assert abs(grad - np.cos(x**2) * 2 * x) < 1e-12
print('✅ 这几行就是 autograd backward 的最小雏形——L54 会把它封装成 Value 类')

## 参数实验：3 层复合的链式法则

构造 3 层复合 h(x)=sin(exp(x²))：
- 最内层 u=x²，导数 du/dx=2x
- 中间层 v=exp(u)，导数 dv/du=exp(u)=exp(x²)
- 外层 y=sin(v)，导数 dy/dv=cos(v)=cos(exp(x²))

链式法则给出：dh/dx = cos(exp(x²)) · exp(x²) · 2x

用步长 `h=1e-7` 的数值差分在 x=[0, 1, -1] 处各算一次数值导数，与上式手算值对比，误差应在 1e-9 以内。这验证了链式法则可直接推广到任意深度的嵌套——autograd 计算图做的正是同一件事。

In [ ]:
# 参数实验：h(x) = sin(exp(x²)) 链式法则验证
# dh/dx = cos(exp(x²)) · exp(x²) · 2x
def h_analytic(x):
    return np.cos(np.exp(x**2)) * np.exp(x**2) * 2 * x

print(f"{'x':>6}  {'解析导数':>14}  {'数值差分':>14}  {'误差':>10}")
print('-' * 50)
for x in [0.0, 1.0, -1.0]:
    analytic = h_analytic(x)
    numerical = (np.sin(np.exp((x+1e-7)**2)) - np.sin(np.exp((x-1e-7)**2))) / 2e-7
    err = abs(analytic - numerical)
    print(f"{x:>6.1f}  {analytic:>14.8f}  {numerical:>14.8f}  {err:>10.2e}")
    assert err < 1e-9, f'x={x}: 误差 {err} 过大'
print('✅ sin(exp(x²)) 的链式法则在三点处均验证通过')


In [ ]:
# 扩展：4 层复合 g(x) = tanh(sin(x²+1))
# dg/dx = (1 - tanh²(sin(x²+1))) · cos(x²+1) · 2x
def g_analytic(x):
    u = x**2 + 1
    v = np.sin(u)
    w = np.tanh(v)
    return (1 - w**2) * np.cos(u) * 2 * x

x = 0.8
analytic = g_analytic(x)
numerical = (np.tanh(np.sin((x+1e-7)**2+1)) - np.tanh(np.sin((x-1e-7)**2+1))) / 2e-7
err = abs(analytic - numerical)
print(f'x={x}: 解析导数={analytic:.8f}, 数值差分={numerical:.8f}, 误差={err:.2e}')
assert err < 1e-5
print('✅ 4 层复合链式法则验证通过')
print('  autograd 做的正是这件事——只是参数空间是百万维，链更深')


## 🎯 未来的回报 (Future Payoff)

今天你亲手拆开的这条「外层导数 × 内层导数」乘法链，是整个深度学习求导的种子。它会在 **L54（autograd）** 变成 `Value.backward()`，并在 **L56（反向传播 backpropagation）** 马上回来——那时整张神经网络的求导，本质上就是把今天这条链沿着计算图一层层连乘。

## 本课收束

现在能调用 `composite_derivative(x)` 得到 `sin(x²)` 在任意点的导数，数值误差在 `1e-9` 量级内。这个"外层导数 × 内层导数"的结构对应 `autograd.backward()`：每个节点缓存局部导数，沿计算图向输入方向连乘。Month 2 实现 autograd 时，`composite_derivative` 的分拆方式会直接变成计算图的边和节点设计。

下一课：**L25**（梯度下降，gradient descent）把今天得到的导数当方向信号，把参数沿负梯度方向推动一步。更远处的 **L54**（autograd）会把这条链式法则直接变成 `Value.backward()`，用计算图自动完成同样的乘法。

---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：链式法则手算（目标 8 分钟）

盖上屏幕，纸上作答：

**问 1**：y = sin(x²)，写出链式法则三步（拆层、内层导数、外层导数、乘积）。  
在 x=1 处，dy/dx = ?（算出数值）

**问 2**：y = (3x+1)⁵，dy/dx = ?  
（提示：外层 f(u)=u⁵，内层 g(x)=3x+1）

**问 3**：h(x) = sin(exp(x²))，写出 dh/dx 的链式法则展开。  
（3层复合，每层各乘一项）

**问 4**：反向传播中，"链式法则"的作用是什么？  
（一句话解释：从最终误差到第 i 层参数梯度的计算路径）

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：y=sin(x²)，x=1 处 dy/dx = cos(1)·2
x1 = 1.0
dy_dx1 = np.cos(x1**2) * 2 * x1
assert np.isclose(dy_dx1, 2 * np.cos(1.0), atol=1e-12)
try:
    cd1 = composite_derivative(x1)
    assert abs(cd1 - dy_dx1) < 1e-7, f"composite_derivative 与解析值不符: {cd1} vs {dy_dx1}"
    print(f"Q1 ✅  dy/dx|_{{x=1}} = cos(1)·2 = {dy_dx1:.6f}，数值={cd1:.6f}")
except (NotImplementedError, TypeError):
    print(f"Q1 ✅  dy/dx|_{{x=1}} = cos(1)·2 = {dy_dx1:.6f}  (composite_derivative 待实现)")

# 问2：y=(3x+1)⁵，dy/dx = 5(3x+1)⁴·3 = 15(3x+1)⁴
f2 = lambda x: (3*x + 1)**5
df2_analytical = lambda x: 15 * (3*x + 1)**4
x2 = 0.5
expected2 = df2_analytical(x2)
numeric2 = (f2(x2 + 1e-7) - f2(x2 - 1e-7)) / (2e-7)
assert np.isclose(numeric2, expected2, rtol=1e-5), f"(3x+1)⁵ 导数不符: {numeric2} vs {expected2}"
print(f"Q2 ✅  d/dx(3x+1)⁵ = 15(3x+1)⁴，在x={x2}处={expected2:.4f}，数值={numeric2:.4f}")

# 问3：h(x)=sin(exp(x²))，dh/dx = cos(exp(x²))·exp(x²)·2x
h = lambda x: np.sin(np.exp(x**2))
dh_analytical = lambda x: np.cos(np.exp(x**2)) * np.exp(x**2) * 2*x
x3 = 0.5
expected3 = dh_analytical(x3)
numeric3 = (h(x3 + 1e-7) - h(x3 - 1e-7)) / (2e-7)
assert np.isclose(numeric3, expected3, rtol=1e-5), f"sin(exp(x²)) 导数不符"
print(f"Q3 ✅  dh/dx=cos(e^x²)·e^x²·2x，在x={x3}处={expected3:.6f}，数值={numeric3:.6f}")

# 问4：反向传播是链式法则的批量执行（验证2层链式法则）
# y=f(g(x)), dy/dx=f'(g(x))·g'(x) — 验证乘法链
g = lambda x: x**2
f_outer = lambda u: np.sin(u)
dg = lambda x: 2*x
df = lambda u: np.cos(u)
x4 = 1.3
chain = df(g(x4)) * dg(x4)
direct_num = (f_outer(g(x4 + 1e-7)) - f_outer(g(x4 - 1e-7))) / (2e-7)
assert np.isclose(chain, direct_num, rtol=1e-6)
print(f"Q4 ✅  链式积 f'(g(x))·g'(x) = {chain:.6f} = 数值差分 {direct_num:.6f}")
print("     BP = 从输出层误差出发，逐层向前乘以该层局部导数，直到参数层")
print("\n🎉 链式法则白板挑战通过！反向传播的数学基础已内化。")

In [ ]:
# ✏️ 本课自评
l24_review = {
    "chain_rule_formula":         None,  # 记住 dy/dx = f'(g(x))·g'(x)？True/False
    "composite_deriv_done":       None,  # composite_derivative 实现并通过断言？True/False
    "3_layer_chain":              None,  # 能展开3层复合的链式乘积？True/False
    "backprop_connection":        None,  # 理解BP=链式法则在计算图上批量执行？True/False
    "whiteboard_passed":          None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l24_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l24_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L24 全部通关！进入 L25：梯度下降')

---

→ **下一课**　[L25 · 梯度下降](L25_gradient_descent.ipynb)

> 下节课将学习 **梯度下降**：蒙眼下山、沿负梯度走到谷底，从损失函数到权重更新公式。